# NB04  Coverage Stress-Test Sample

Builds a **balanced 96-company grid** (4 size tiers x 3 sectors x 8 companies) for stress-testing news-API coverage.

> This sample is **non-representative by construction.** You should  not reuse it as a population estimate.

### Why balanced grid VS proportional sample?

The goal here is to measure how well the news APIs cover companies across different **size x sector combinations**, not to produce a miniature replica of the population.

A proportional draw from the 1.37M-company dataset would allocate roughly according to the population distribution: `Medium` is only ~0.1% of the population, so at n≈100 it would round to ~0 companies. You can't measure a coverage rate on 0 companies, yet that's exactly the cell that matters for understanding whether the API surfaces news on medium-sized firms.

Instead, I draw **equal numbers from each cell** (8 per cell, 12 cells = 96 total), so every sizexsector combination carries the same measurable signal. The trade-off is that the sample is skewed relative to the real population.

The 4 size tiers used here (`Micro / Small / Medium / Large`) come from the same `Accounts.AccountCategory` → size-tier mapping established in NB01. The 5 filing-status buckets (`Dormant`, `No Filings`, `Subsidiary` x 2, `Unknown`) are **excluded**  they are not size indicators and would contaminate the coverage measurement.

### 1. Macro variables

I start by defining some macro variables that can then be edited, this is a design demo:
1. `_SIZE_TIERS` maps the four genuine size tiers to keep. The filing-status buckets (`Dormant`, `No Filings`, `Subsidiary`, `Unknown`) are implicitly excluded by this whitelist.
2. A list of **legal suffixes** (`_LEGAL_SUFFIX_RE`) to strip from company names (for now manually sourced; later we could run pattern analysis to catch edge cases).
3. A set of **query templates** (`_QUERY_TEMPLATES`) customised per sector, used to build targeted news-API search strings.

In [1]:
import re
import pandas as pd

_SIZE_TIERS = {"Micro", "Small", "Medium", "Large"}

_LEGAL_SUFFIX_RE = re.compile(
    r"\s+(LIMITED|LTD|PLC|LLP|L\.T\.D\.)\.?\s*$", re.IGNORECASE
)

_QUERY_TEMPLATES = {
    "Technology, legal & professional": [
        "{name} contract",
        "{name} acquisition",
        "{name} revenue",
        "{name} partnership",
        "{name} funding",
    ],
    "Fast growth & emerging": [
        "{name} funding round",
        "{name} investment",
        "{name} series",
        "{name} growth",
        "{name} startup",
    ],
    "Manufacturing": [
        "{name} supply chain",
        "{name} production",
        "{name} export",
        "{name} acquisition",
        "{name} contract",
    ],
}

### 2. Helper functions

I create 2 lightweight functions to handle all the transformation logic. I've kept them here in the notebook rather than in a separate module so the notebook is fully self-contained + we can reuse them when building an app/dashboard.

1. `normalise_search_name` builds a clean search string for news-API queries. It strips common legal suffixes (`LIMITED`, `LTD`, `PLC`, `LLP`, `L.T.D.`) which add noise to searches, lowercases the result, and appends the town and sector keyword. For example: `"ACME ENGINEERING LIMITED"` + `"BIRMINGHAM"` + `"Manufacturing"` == `"acme engineering birmingham manufacturing"`. If `PostTown` is missing (NaN), it's silently dropped rather than producing a double space.

2. **`build_stress_test_sample`** does the actual sampling. It filters the population to the four size tiers using the `segment` column, then draws `n_per_cell` companies from each `segment x sector` group using a seeded random sample. It also attaches `search_name` and `query_templates` to every row so downstream notebook cells have everything they need without rejoining.

In [2]:
def normalise_search_name(company_name, post_town, sector):
    name = _LEGAL_SUFFIX_RE.sub("", company_name).strip().lower()
    sector_keyword = sector.lower().replace(",", "")
    town = post_town.lower() if isinstance(post_town, str) else ""
    return " ".join(part for part in [name, town, sector_keyword] if part)


def build_stress_test_sample(df, n_per_cell=8, seed=42):
    df = df[df["segment"].isin(_SIZE_TIERS)].copy()

    frames = [
        group.sample(n=min(n_per_cell, len(group)), random_state=seed)
        for _, group in df.groupby(["segment", "sector"])
    ]
    cells = pd.concat(frames, ignore_index=True)

    cells["search_name"] = cells.apply(
        lambda r: normalise_search_name(r["CompanyName"], r["RegAddress.PostTown"], r["sector"]),
        axis=1,
    )
    cells["query_templates"] = cells["sector"].map(
        lambda s: [t.format(name="{name}") for t in _QUERY_TEMPLATES.get(s, [])]
    )
    return cells.reset_index(drop=True)

In [3]:
SOURCE = '../data/processed/filtered_bb_sme_sectors.csv'
RANDOM_SEED = 42
N_PER_CELL = 8
SEARCH_DATE = "2026-06-23"  # pin for reproducibility

population = pd.read_csv(SOURCE, low_memory=False)
print(f"Population: {len(population):,} rows")

Population: 1,372,321 rows


### 3. Configuration and data load

`RANDOM_SEED` and `SEARCH_DATE` are pinned at the top for reproducibility. Changing `RANDOM_SEED` produces a different 96-company draw; `SEARCH_DATE` is passed downstream to date-bound news queries so results are anchored to the same window regardless of when the notebook is re-run.

The source is the filtered population from NB01.

In [4]:
sample = build_stress_test_sample(population, n_per_cell=N_PER_CELL, seed=RANDOM_SEED)
print(f"Sample: {len(sample)} rows")
sample.head()

Sample: 96 rows


,CompanyName,CompanyNumber,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,PreviousName_9.CONDATE,PreviousName_9.CompanyName,PreviousName_10.CONDATE,PreviousName_10.CompanyName,ConfStmtNextDueDate,ConfStmtLastMadeUpDate,sector,segment,search_name,query_templates
0,WHALAR LTD,09803195,NaN,NaN,9TH FLOOR,107 CHEAPSIDE,LONDON,NaN,NaN,EC2V 6DN,...,NaN,NaN,NaN,NaN,14/10/2026,30/09/2025,Fast growth & emerging,Large,whalar london fast growth & emerging,"[{name} funding round, {name} investment, {nam..."
1,NCB RIP,15336152,NaN,NaN,23 MENTMORE TERRACE,NaN,LONDON,NaN,ENGLAND,E8 3PN,...,NaN,NaN,NaN,NaN,10/10/2026,26/09/2025,Fast growth & emerging,Large,ncb rip london fast growth & emerging,"[{name} funding round, {name} investment, {nam..."
2,EXODUSPOINT CAPITAL MANAGEMENT UK TECHNOLOGIES...,12329778,NaN,NaN,20 ST. JAMES'S STREET,NaN,LONDON,NaN,UNITED KINGDOM,SW1A 1ES,...,NaN,NaN,NaN,NaN,04/03/2027,18/02/2026,Fast growth & emerging,Large,exoduspoint capital management uk technologies...,"[{name} funding round, {name} investment, {nam..."
3,AKSIA EUROPE LIMITED,06890595,NaN,NaN,20 ST. JAMES'S STREET,NaN,LONDON,NaN,ENGLAND,SW1A 1ES,...,NaN,NaN,NaN,NaN,25/09/2026,11/09/2025,Fast growth & emerging,Large,aksia europe london fast growth & emerging,"[{name} funding round, {name} investment, {nam..."
4,CME TECHNOLOGY AND SUPPORT SERVICES LIMITED,NI610336,NaN,NaN,25 GREAT VICTORIA STREET,NaN,BELFAST,NaN,NORTHERN IRELAND,BT2 7AQ,...,NaN,NaN,NaN,NaN,26/12/2026,12/12/2025,Fast growth & emerging,Large,cme technology and support services belfast fa...,"[{name} funding round, {name} investment, {nam..."


### 4. Build the sample

`build_stress_test_sample` runs the full pipeline in one call: derives size tiers, filters excluded categories, draws 8 companies per cell, and attaches `search_name` and `query_templates` to each row. The result is a flat 96-row DataFrame ready for news-API querying.

In [5]:
pivot = sample.groupby(['segment', 'sector']).size().unstack(fill_value=0)
print("Companies per cell (should all be 8):")
pivot

Companies per cell (should all be 8):


sector,Fast growth & emerging,Manufacturing,"Technology, legal & professional"
segment,,,
Large,8,8,8
Medium,8,8,8
Micro,8,8,8
Small,8,8,8


#### 4.1 Validate cell balance

A quick pivot confirms all 12 cells are present and each has exactly 8 companies. If any cell shows a number other than 8 it means that cell had fewer than 8 companies in the population (unlikely but worth checking if the source data is ever re-filtered upstream).

In [6]:
OUT = '../data/processed/nb04_stress_test_sample.csv'
sample.to_csv(OUT, index=False)
print(f"Saved → {OUT}")

Saved → ../data/processed/nb04_stress_test_sample.csv
